# Read Binary Files Lazily With Xarray and Dask

This notebook shows the main xarray-binfile workflow:

- describe a binary-file convention with `FileSpecsGetter`
- open individual files or a whole collection with `engine="binfile"`
- keep the resulting arrays lazy and Dask-backed
- inspect the task graph before executing a calculation
- call `.compute()` only when you want concrete results in memory

In [ ]:
import pathlib
import re
import tempfile
from shutil import which

import numpy as np
import xarray as xr
from IPython.display import Markdown, display

import xarray_binfile.write as xbf_write
from xarray_binfile.tutorial import DatasetGenerator, FileSpecsGetter

_ = xbf_write

temporary_directory = tempfile.TemporaryDirectory()
data_directory = pathlib.Path(temporary_directory.name)
data_directory

In [ ]:
base_coords = {
    "x": np.linspace(0.0, 3.0, num=32, dtype=np.float32),
    "y": np.linspace(-1.0, 1.0, num=24, dtype=np.float32),
    "z": np.linspace(0.0, 2.0, num=16, dtype=np.float32),
}

file_specs_getter = FileSpecsGetter(
    base_coords=base_coords,
    dtype=np.float32,
    filename_template="{name}-{digits:04}.bin",
    filename_regex=re.compile(r"(?P<name>\w+)-(?P<digits>\d{4})\.bin"),
)

dataset_generator = DatasetGenerator(file_specs_getter.reader)
filenames = (
    file_specs_getter.filename_template.format(name=name, digits=step)
    for name in ("ux", "uy", "uz")
    for step in range(12)
)
source_dataset = dataset_generator(map(pathlib.Path, filenames))
source_dataset.binary_engine.to_file(file_specs_getter.writer, data_directory)
sorted(path.name for path in data_directory.glob("*.bin"))[:6]

## Open one file

The backend can read a single raw binary file into a small `Dataset`. The metadata getter supplies the dtype, coordinates, and variable name that the binary file itself does not store.

In [ ]:
single_dataset = xr.open_dataset(
    sorted(data_directory.glob("ux-*.bin"))[0],
    engine="binfile",
    read_specs_getter=file_specs_getter.reader,
)
single_dataset

## Open many files lazily

`xr.open_mfdataset` combines the files by coordinates and keeps the arrays lazy because we pass Dask chunks. The example data here is intentionally small enough for a fast docs build, but the same chunked pattern is the one you use when the full dataset no longer fits comfortably in memory.

In [ ]:
lazy_dataset = xr.open_mfdataset(
    sorted(data_directory.glob("*.bin")),
    engine="binfile",
    read_specs_getter=file_specs_getter.reader,
    chunks={"x": 8, "y": 6, "z": 4, "time": 3},
    parallel=True,
)
lazy_dataset

In [ ]:
ux = lazy_dataset["ux"]
print(type(ux.data))
print(ux.chunks)
print(
    f"Estimated in-memory size if fully loaded: {lazy_dataset.nbytes / 1024**2:.2f} MiB"
)
ux.data

The `dask.array` representation is the important part here. At this stage the notebook has built a task graph, not loaded every chunk into memory. That is what lets the same workflow scale to much larger datasets.

In [ ]:
lazy_speed = np.sqrt(
    lazy_dataset["ux"] ** 2 + lazy_dataset["uy"] ** 2 + lazy_dataset["uz"] ** 2
).rename("speed")
lazy_mean_speed = lazy_speed.mean(dim="time")
lazy_mean_speed

## Visualize the Dask task graph

This graph represents the operations that will be executed when the result is finally computed. It is still a plan at this point, not a loaded array.

In [ ]:
if which("dot"):
    display(lazy_mean_speed.data.visualize(rankdir="LR", format="svg"))
else:
    display(
        Markdown(
            "Graphviz is not available in this environment, so the task graph image cannot be rendered here. "
            "The published documentation installs Graphviz and renders this cell as an SVG task graph."
        )
    )

## Execute the calculation

Calling `.compute()` turns the lazy Dask graph into actual work and materializes the final result in memory.

In [ ]:
computed_mean_speed = lazy_mean_speed.compute()
computed_mean_speed

In [ ]:
display(computed_mean_speed.sel(x=slice(0.5, 2.5), y=slice(-0.5, 0.5)).isel(z=0))
computed_mean_speed.isel(z=0).plot(cmap="viridis", robust=True)

The rest of the analysis story comes from xarray and Dask themselves. For more on indexing, plotting, grouped operations, distributed execution, and storage backends, see the [xarray documentation](https://docs.xarray.dev) and the [Dask documentation](https://docs.dask.org).